In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from sklearn import datasets
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_openml

%matplotlib inline

In [ ]:
import os

# 創建data資料夾（如果不存在）
data_dir = 'data'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"創建資料夾: {data_dir}")

# 設置MNIST數據的緩存目錄
cache_dir = os.path.join(data_dir, 'openml_cache')

# 載入MNIST數據集並存放在data資料夾中
print("正在載入MNIST數據集...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto', data_home=cache_dir)
X_mnist, y_mnist = mnist.data, mnist.target.astype(int)

print(f"MNIST數據已存放在: {cache_dir}")

# 只選擇數字4和數字7
mask = (y_mnist == 4) | (y_mnist == 7)
X_selected = X_mnist[mask]
y_selected = y_mnist[mask]

# 將標籤轉換為二進制 (4->0, 7->1)
y_binary = (y_selected == 7).astype(int)

print(f"原始數據形狀: {X_selected.shape}")
print(f"標籤分布: {np.bincount(y_binary)}")
print(f"數字4的樣本數: {np.sum(y_binary == 0)}")
print(f"數字7的樣本數: {np.sum(y_binary == 1)}")

In [ ]:
# 使用PCA將784維降到2維
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_selected)

print('降維後數據形狀:', X_pca.shape)
print('PCA解釋的變異比例:', pca.explained_variance_ratio_)
print('累積解釋變異比例:', np.sum(pca.explained_variance_ratio_))
print('Class labels:', np.unique(y_binary))

# 可視化PCA後的數據分布
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[y_binary == 0, 0], X_pca[y_binary == 0, 1], 
           c='red', marker='s', label='Digit 4', alpha=0.7)
plt.scatter(X_pca[y_binary == 1, 0], X_pca[y_binary == 1, 1], 
           c='blue', marker='x', label='Digit 7', alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.3f})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.3f})')
plt.title('MNIST Digits 4 & 7 after PCA')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 分割數據為70%訓練和30%測試
X_train, X_test, y_train, y_test = train_test_split(X_pca, y_binary, test_size=0.3, random_state=42, stratify=y_binary)
print('Labels count in y:', np.bincount(y_binary))
print('Labels count in y_train:', np.bincount(y_train))
print('Labels count in y_test:', np.bincount(y_test))

In [ ]:
# 標準化特徵
sc = StandardScaler()
sc.fit(X_train)
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

print("訓練數據標準化後的統計:")
print(f"平均值: {np.mean(X_train_std, axis=0)}")
print(f"標準差: {np.std(X_train_std, axis=0)}")

In [ ]:
class LogisticRegressionMSE:
    """使用MSE作為損失函數的邏輯回歸"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.costs = []
    
    def sigmoid(self, z):
        # 避免溢出
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        # 初始化權重
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # 梯度下降
        for i in range(self.n_iterations):
            # 前向傳播
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)
            
            # MSE損失
            cost = np.mean((y_pred - y) ** 2)
            self.costs.append(cost)
            
            # 計算梯度
            dw = (2/n_samples) * np.dot(X.T, (y_pred - y))
            db = (2/n_samples) * np.sum(y_pred - y)
            
            # 更新權重
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
    
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

class LogisticRegressionCrossEntropy:
    """使用Cross-Entropy作為損失函數的邏輯回歸"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.costs = []
    
    def sigmoid(self, z):
        # 避免溢出
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        # 初始化權重
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # 梯度下降
        for i in range(self.n_iterations):
            # 前向傳播
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)
            
            # Cross-Entropy損失
            # 避免log(0)
            y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
            cost = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
            self.costs.append(cost)
            
            # 計算梯度
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # 更新權重
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
    
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
# 訓練兩個模型
lr_mse = LogisticRegressionMSE(learning_rate=0.1, n_iterations=1000)
lr_ce = LogisticRegressionCrossEntropy(learning_rate=0.1, n_iterations=1000)

lr_mse.fit(X_train_std, y_train)
lr_ce.fit(X_train_std, y_train)

# 預測和評估
y_pred_mse = lr_mse.predict(X_test_std)
y_pred_ce = lr_ce.predict(X_test_std)

print("MSE Loss 模型準確率:", accuracy_score(y_test, y_pred_mse))
print("Cross-Entropy Loss 模型準確率:", accuracy_score(y_test, y_pred_ce))

In [ ]:
# 創建損失函數的3D可視化
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def mse_loss(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def cross_entropy_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# 創建特徵空間的網格
feature1_range = np.linspace(X_train_std[:, 0].min() - 1, X_train_std[:, 0].max() + 1, 30)
feature2_range = np.linspace(X_train_std[:, 1].min() - 1, X_train_std[:, 1].max() + 1, 30)
F1, F2 = np.meshgrid(feature1_range, feature2_range)

# 計算每個點的損失值
mse_losses = np.zeros(F1.shape)
ce_losses = np.zeros(F1.shape)

for i in range(F1.shape[0]):
    for j in range(F1.shape[1]):
        # 創建測試點
        test_point = np.array([[F1[i, j], F2[i, j]]])
        
        # 使用訓練好的模型預測
        prob_mse = lr_mse.predict_proba(test_point)[0]
        prob_ce = lr_ce.predict_proba(test_point)[0]
        
        # 計算該點對所有訓練數據的平均損失
        mse_loss_val = 0
        ce_loss_val = 0
        
        for k in range(len(y_train)):
            mse_loss_val += (prob_mse - y_train[k]) ** 2
            ce_loss_val += -(y_train[k] * np.log(np.clip(prob_mse, 1e-15, 1-1e-15)) + 
                           (1 - y_train[k]) * np.log(np.clip(1 - prob_mse, 1e-15, 1-1e-15)))
        
        mse_losses[i, j] = mse_loss_val / len(y_train)
        ce_losses[i, j] = ce_loss_val / len(y_train)

In [ ]:
# 創建3D圖形
fig = plt.figure(figsize=(16, 6))

# MSE Loss 3D 圖
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(F1, F2, mse_losses, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Feature 1 (Petal Length)')
ax1.set_ylabel('Feature 2 (Petal Width)')
ax1.set_zlabel('MSE Loss')
ax1.set_title('MSE Loss Surface')
fig.colorbar(surf1, ax=ax1, shrink=0.5)

# Cross-Entropy Loss 3D 圖
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(F1, F2, ce_losses, cmap='plasma', alpha=0.8)
ax2.set_xlabel('Feature 1 (Petal Length)')
ax2.set_ylabel('Feature 2 (Petal Width)')
ax2.set_zlabel('Cross-Entropy Loss')
ax2.set_title('Cross-Entropy Loss Surface')
fig.colorbar(surf2, ax=ax2, shrink=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 比較訓練過程中的損失變化
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(lr_mse.costs, label='MSE Loss', color='blue')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('MSE Loss During Training')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(lr_ce.costs, label='Cross-Entropy Loss', color='red')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Cross-Entropy Loss During Training')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 繪製決策邊界比較
def plot_decision_regions(X, y, classifier, test_idx=None, resolution=0.02):
    # 設置標記和顏色
    markers = ('s', 'x', 'o', '^', 'v')
    colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[:len(np.unique(y))])
    
    # 繪製決策面
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                          np.arange(x2_min, x2_max, resolution))
    Z = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    Z = Z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())
    
    # 繪製所有樣本
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(x=X[y == cl, 0], y=X[y == cl, 1],
                   alpha=0.8, c=colors[idx],
                   marker=markers[idx], label=cl, edgecolor='black')
    
    # 突出顯示測試樣本
    if test_idx:
        X_test, y_test = X[test_idx, :], y[test_idx]
        plt.scatter(X_test[:, 0], X_test[:, 1],
                   c='black', alpha=1.0, linewidths=2,
                   marker='o', s=100, label='test set')

# 合併訓練和測試數據來繪製完整圖像
X_combined_std = np.vstack((X_train_std, X_test_std))
y_combined = np.hstack((y_train, y_test))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_decision_regions(X_combined_std, y_combined, lr_mse, 
                     test_idx=range(len(y_train), len(y_train) + len(y_test)))
plt.xlabel('Petal Length [standardized]')
plt.ylabel('Petal Width [standardized]')
plt.title('MSE Loss - Decision Boundary')
plt.legend(loc='upper left')

plt.subplot(1, 2, 2)
plot_decision_regions(X_combined_std, y_combined, lr_ce,
                     test_idx=range(len(y_train), len(y_train) + len(y_test)))
plt.xlabel('Petal Length [standardized]')
plt.ylabel('Petal Width [standardized]')
plt.title('Cross-Entropy Loss - Decision Boundary')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 印出模型參數比較
print("模型參數比較:")
print("=" * 50)
print("MSE Loss 模型:")
print(f"  權重: {lr_mse.weights}")
print(f"  偏置: {lr_mse.bias}")
print(f"  最終損失: {lr_mse.costs[-1]:.6f}")

print("\nCross-Entropy Loss 模型:")
print(f"  權重: {lr_ce.weights}")
print(f"  偏置: {lr_ce.bias}")
print(f"  最終損失: {lr_ce.costs[-1]:.6f}")

print("\n性能比較:")
print("=" * 50)
print(f"MSE Loss 模型準確率: {accuracy_score(y_test, y_pred_mse):.4f}")
print(f"Cross-Entropy Loss 模型準確率: {accuracy_score(y_test, y_pred_ce):.4f}")